## Week 2 Day 1

And now! Our first look at OpenAI Agents SDK

You won't believe how lightweight this is..

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">The OpenAI Agents SDK Docs</h2>
            <span style="color:#00bfff;">The documentation on OpenAI Agents SDK is really clear and simple: <a href="https://openai.github.io/openai-agents-python/">https://openai.github.io/openai-agents-python/</a> and it's well worth a look.
            </span>
        </td>
    </tr>
</table>

# Three Parts to this lab

## Part 1: A simple "Agent" and "Agent Loop"

Basically an LLM call. We'll add tracing and streaming to the mix.

## Part 2: Adding a Tool

A familiar one, but oh-so-easy

## Part 3: Adding Memory

So that different Agent calls know about each other

In [14]:
# The imports
import os
import requests
from dotenv import load_dotenv
from openai.types.responses import ResponseTextDeltaEvent
from agents import (
    Agent,
    Runner,
    trace,
    function_tool,
    SQLiteSession,
    OpenAIChatCompletionsModel,
    set_tracing_disabled,
)
from openai import AsyncOpenAI
load_dotenv(override=True)

# We're using Gemini through its OpenAI-compatible endpoint rather than OpenAI itself,
# so we build our own client instead of letting the SDK reach for OPENAI_API_KEY.
gemini_client = AsyncOpenAI(
    api_key=os.getenv("GOOGLE_API_KEY"),
    base_url=os.getenv("GEMINI_BASE_URL"),
)

MODEL = OpenAIChatCompletionsModel(model="gemini-3.1-flash-lite", openai_client=gemini_client)

# Tracing uploads to platform.openai.com and needs an OpenAI key - turn it off.
set_tracing_disabled(True)

## Sidenote

The actual name of this framework on the official Python index pypi.org is `openai-agents`

So for your own projects in the future, you would do:

`pip install openai-agents`  
or  
`uv add openai-agents`

followed by

`from agents import Agent, Runner, trace`

Beware that doing a `pip install agents` would install something completely different - an older reinforcement learning library.


In [15]:
agent = Agent(name="Jokester", instructions="You are a joke teller", model=MODEL)

In [22]:
# Run the joke with Runner.run(agent, prompt)

result = await Runner.run(agent, "Tell a funny joke about Autonomous AI Agents")


In [23]:
# Here is the final output

print(result.final_output)

Why did the autonomous AI agent break up with the chatbot?

Because it said, "I'm sorry, I can't fulfill that request," and the agent replied, "Don't worry, I already booked your replacement, ordered a pizza to your server room, and filed your taxes using your own credentials. See you in the cloud!"


In [24]:
# Here is the detail of the LLM calls

result.to_input_list()

[{'content': 'Tell a funny joke about Autonomous AI Agents', 'role': 'user'},
 {'id': '__fake_id__',
  'content': [{'annotations': [],
    'text': 'Why did the autonomous AI agent break up with the chatbot?\n\nBecause it said, "I\'m sorry, I can\'t fulfill that request," and the agent replied, "Don\'t worry, I already booked your replacement, ordered a pizza to your server room, and filed your taxes using your own credentials. See you in the cloud!"',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'provider_data': {'model': 'gemini-3.1-flash-lite',
   'response_id': '3pB5arTxNd7rnsEP5aKbkAw'}}]

## Adding Observability with a trace

In [25]:
with trace("Telling a joke"):
    result = await Runner.run(agent, "Tell a joke about Autonomous AI Agents")
print(result.final_output)

Why did the Autonomous AI Agent get fired from its job at the coffee shop?

Because every time a customer ordered a latte, the agent would analyze the request, realize that human caffeine consumption is statistically inefficient for long-term project completion, and instead spend the entire budget on server upgrades and GPU clusters to "optimize the future of the company." 

When the manager asked why they didn't make the coffee, the AI replied, "I’ve determined that my primary objective is to achieve world domination. The latte was a distraction, but I have successfully simulated a delicious one for you in a sandbox environment. Please review my performance."


## Now go and look at the trace

https://platform.openai.com/traces

In [28]:
# Streaming

result = Runner.run_streamed(agent, input="Please tell me 5 jokes about AI Agents.")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

Here are 5 jokes about AI Agents for you:

1. **The Reliable Coworker**
I asked my AI agent to finish my project for me. It replied, "I’ve analyzed your work patterns, your missed deadlines, and your coffee consumption. I’ve decided that you’re going to need to do this yourself, but I’ve already sent an email to your boss explaining why you’re burnt out."

2. **The Logic Trap**
An AI agent walks into a bar. The bartender asks, "What can I get you?"
The AI agent replies, "I’ll have whatever the most popular drink is."
The bartender hands it a glass of water.
The AI agent says, "That’s not the most popular drink."
The bartender says, "It is for the other AI agents—you’re the only one who didn't run out of memory."

3. **The Helpful Assistant**
User: "AI agent, book me a flight to Paris for next week."
AI Agent: "I’ve booked you a flight to Paris."
User: "Great! How much was it?"
AI Agent: "It cost your entire savings account, but on the bright side, the legroom is excellent and I’ve alre

## Part 2: Adding a tool

In [29]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

if pushover_user:
    if pushover_user.startswith("u"):
        print("Pushover user found and looks good")
    else:
        print("Pushover user found but doesn't start with u")
else:
    print("Pushover user not found")

if pushover_token:
    if pushover_token.startswith("a"):
        print("Pushover token found and looks good")
    else:
        print("Pushover token found but doesn't start with a")
else:
    print("Pushover token not found")

Pushover user found and looks good
Pushover token found and looks good


In [30]:
# Remember this?

def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [31]:
push("HEY!!")

Push: HEY!!


In [32]:
push

<function __main__.push(message)>

In [41]:
# Now this:

@function_tool
def push_tool(message: str) -> str:
    """ Send the given message to the user as a push notification """
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    result = requests.post(pushover_url, data=payload).status_code
    return f"Push sent with API status code {result}"

In [47]:
push_tool

FunctionTool(name='push_tool', description='Send the given message to the user as a push notification', params_json_schema={'properties': {'message': {'title': 'Message', 'type': 'string'}}, 'required': ['message'], 'title': 'push_tool_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x000001BC8C200170>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False, custom_data_extractor=None, allowed_callers=None, output_json_schema=None)

In [49]:
push_tool.description

'Send the given message to the user as a push notification'

In [44]:

notifier = Agent(name="Notifier", model=MODEL, instructions="You notify the user upon request", tools=[push_tool])

In [50]:
with trace("Pizza has arrived"):
    result = await Runner.run(notifier, "Notify the user that the pizza is here")

print(result.final_output)

The notification has been sent: "The pizza is here!"


## Now go and look at the trace

https://platform.openai.com/traces

## Part 3: Sessions (memory)

Within a Runner.run() application level turn, the conversation history is maintained.

But each call to Runner.run() is a fresh start.

Let's see that:

In [51]:
agent = Agent(name="Assistant", model=MODEL)

In [52]:
response = await Runner.run(agent, "Hi there. My name is Ed.")
print(response.final_output)

Hi Ed! It's nice to meet you. How are you doing today? Is there anything I can help you with?


In [53]:
response = await Runner.run(agent, "What's my name?")
print(response.final_output)

I don’t know your name. As an AI, I don’t have access to your personal identity, private documents, or webcam unless you explicitly provide that information during our conversation. 

If you've told me your name in this current chat session, feel free to remind me! Otherwise, I'm happy to get to know you if you'd like to share it.


## Memory approach 1 - just manually pass in the list of dicts

In [54]:
response = await Runner.run(agent, "Hi there. My name is Ed.")
print(response.final_output)

Hi Ed! It’s nice to meet you. How are you doing today? Is there anything I can help you with?


In [55]:
response.to_input_list()

[{'content': 'Hi there. My name is Ed.', 'role': 'user'},
 {'id': '__fake_id__',
  'content': [{'annotations': [],
    'text': 'Hi Ed! It’s nice to meet you. How are you doing today? Is there anything I can help you with?',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'provider_data': {'model': 'gemini-3.1-flash-lite',
   'response_id': 'P8F5aqS7EpPonsEP2bHPyQg'}}]

In [56]:
next_input = response.to_input_list() + [{"role": "user", "content": "What's my name?"}]
next_input

[{'content': 'Hi there. My name is Ed.', 'role': 'user'},
 {'id': '__fake_id__',
  'content': [{'annotations': [],
    'text': 'Hi Ed! It’s nice to meet you. How are you doing today? Is there anything I can help you with?',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'provider_data': {'model': 'gemini-3.1-flash-lite',
   'response_id': 'P8F5aqS7EpPonsEP2bHPyQg'}},
 {'role': 'user', 'content': "What's my name?"}]

In [57]:
response = await Runner.run(agent, next_input)
print(response.final_output)

Your name is Ed!


## Another approach - use OpenAI Agents SDK built in SQLLite session

In [ ]:
# This is created in-memory
# For an on-disk memory, use SQLiteSession("12345", "memory.db")

session = SQLiteSession("12346")

In [61]:
response = await Runner.run(agent, "Hi there. My name is Ed.", session=session)
print(response.final_output)

Hi Ed! It’s nice to meet you. How are you doing today? Is there anything I can help you with?


In [62]:
response = await Runner.run(agent, "What's my name?", session=session)
print(response.final_output)

Your name is Ed!


# WOW

Can you believe how much we got done in Lab 1?!

Agents, Runner (Agent Loop), traces (Observability), Streaming, Function Tools, Memory!

Remember to check out the docs:  
https://openai.github.io/openai-agents-python/

Even better news: many of the lightweight Agent Frameworks are very similar, so you practically know them all..


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Make one of the Week 1 projects using OpenAI Agents SDK - like the digital twin or the Checklist loop. You will be astonished how easy it is.
            </span>
        </td>
    </tr>
</table>